# PSA_Calibration: 位置の補正、ピクセル、エネルギー

バタフライマップの歪んだ格子を、まっすぐな 8 × 8 の格子に直して、ヒットごとのピクセル (n, m) を決めます。
`grid_spect.py` が作る補正を `pixel_id.py` で当てるのと同じ計算です。

| 手順 | 中身 |
|---|---|
| 1. 64 点 | `calib/psa/position_haNN.csv`(`peak_search.py` が見つけたバタフライマップのピーク、ch ごとに 64 点 `x, y`)を読む |
| 2. (n, m) | 64 点にピクセル番号を付ける: x で並べて 8 個ずつ 8 列(n = 1〜8)、列の中で y で並べる(m = 1〜8)。ファイルに `n, m` の列があればそれを使う |
| 3. 補正 | 64 点 (x, y) → (n, m) を通る**薄板スプライン**(thin-plate spline、平滑化なし)を ch ごとに作り、ヒットの (`x_hg`, `y_hg`) を (`x_corr`, `y_corr`) に移す |
| 4. ピクセル | $n - 0.5 \le$ `x_corr` $< n + 0.5$、m も同じ。8 × 8 の外は `pix_n = pix_m = -1`。`pixel = 64 ch + 8 (n − 1) + (m − 1)`(0〜383、外は −1) |
| 5. 温度 | `calib/psa/temp_gain_haNN.csv`(ch ごとの温度と相対ゲインの表)で `hg_corr = hg / gain(T)`。T はボードの温度センサー 6 個の平均 |
| 6. エネルギー | `calib/psa/energy_haNN.csv` のピクセルごとの `p0, p1` で `energy = p0 + p1 × hg_corr` [keV] |

薄板スプラインは $f(\mathbf p) = \sum_k w_k\, \phi(|\mathbf p - \mathbf p_k|) + c_0 + c_1 x + c_2 y$、$\phi(r) = r^2 \log r$ で、
64 点をちょうど通ります(scipy の `RBFInterpolator(kernel="thin_plate_spline", smoothing=0)` と同じ関数。`ThinPlateSpline.cxx`)。

`n, m` の付け方で、x や y がまったく同じピークがあると、C++ と numpy で並びが変わることがあります。
そのため `run_script/psa_position_nm.py` で numpy の決めた `n, m` をファイルに書き込んでおきます(`config/README.md` の「PSA の較正ファイル」)。
同じデータで `pixel_id.py` とピクセルが全部一致することは、`scripts/check/compare_psa_python.py` で確かめられます。

較正ファイルの作り方は [4_MakeCalibration](../../../4_MakeCalibration/) と `config/README.md` の「PSA の較正ファイル」にあります。

> **今の較正ファイル**(2026-11-03 の UVSOR のデータから作ったもの):
> 位置は Co の run12 から `--method grid` で、温度ゲインとエネルギー(Cs-137 の 662 keV、線 1 本)は Cs の run01〜11 から。
> ha04 は run12 と Cs のランにほとんど無いので、位置は前の仮のもの(peak_search の結果)で、温度ゲインとエネルギーはありません。
> ha01 の ch4 はバタフライマップがにじんでいて、位置の較正を作っていません(ピクセルなし)。

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import ROOT
ROOT.gROOT.SetBatch(True)

# リポジトリの場所、チェーンの実行スクリプト、設定ファイル(config/*.json)の読み込み
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_script" / "run_chain.py").exists())
RUN_CHAIN = REPO / "run_script" / "run_chain.py"
sys.path.insert(0, str(REPO / "config"))
import ericanext_config as ec
CACHE = REPO / "tutorials" / "cache"   # チェーンの出力の置き場(チュートリアル間で共有、git には入れない)
CACHE.mkdir(exist_ok=True)

def th2(h):
    """ROOT の TH2 -> (中身 [x のビン, y のビン], x のビンの端, y のビンの端)"""
    nx, ny = h.GetNbinsX(), h.GetNbinsY()
    a = np.frombuffer(h.GetArray(), dtype=np.float64, count=(nx + 2) * (ny + 2)).reshape(ny + 2, nx + 2)
    ex = np.array([h.GetXaxis().GetBinLowEdge(i) for i in range(1, nx + 2)])
    ey = np.array([h.GetYaxis().GetBinLowEdge(i) for i in range(1, ny + 2)])
    return a[1:-1, 1:-1].T.copy(), ex, ey

def show_params(config, module):
    """設定ファイルの、あるモジュールのパラメータを、説明(_doc)と単位(_units)つきで表示する"""
    p = json.load(open(REPO / "config" / f"{config}.json"))["parameters"][module]
    doc, units = p.get("_doc", {}), p.get("_units", {})
    for k, v in p.items():
        if k.startswith("_"):
            continue
        u = f" [{units[k]}]" if k in units else ""
        print(f"{k} = {json.dumps(v, ensure_ascii=False)}{u}\n    {doc.get(k, '')}")

## 1. データの準備

2026-11-03 の UVSOR での **HA_cal のラン `20261103_run01`**(PSA だけ、14 ボード、約 840 MB)を使います。
位置の較正(`calib/psa/position_haNN.csv`)を作ったときと同じ日の較正データです。
データは解析用の計算機の D ドライブ(`/mnt/d/UVSOR2026_late`)にあります。
ほかの場所にあるときは、環境変数 `SMILE3_PSA_CAL_DATA` に `UVSOR2026_late` に当たるディレクトリを設定してください。

1 つのランのボードは、2 つの storage に分かれて書かれています。

| storage | ディレクトリ | PSA のボード |
|---|---|---|
| storage_g1 | `killua/<ラン>` | ha00〜ha05 |
| storage_g0 | `calltt/<ラン>` | ha06〜ha16 |

`run_chain.py` には、片方を `--data-dir`、もう片方を `PSA_ReadData.extra_data_dirs` で渡します。

In [ ]:
DATA = Path(os.environ.get("SMILE3_PSA_CAL_DATA", "/mnt/d/UVSOR2026_late"))
RUN = "20261103_run01"
RUN_DIRS = [DATA / "killua" / RUN, DATA / "calltt" / RUN]
for d in RUN_DIRS:
    print("OK     " if d.is_dir() else "MISSING", d)

## 2. チェーンを流す

設定ファイル `smile3_psa`(`PSA_ReadData → PSA_CalcFADC → PSA_CenterGravity → PSA_Calibration → PSA_TreeWriter`)で、ラン全体を流します(40 秒ほど)。
ターミナルでは、リポジトリの `run_script` で次を実行するのと同じです。

```bash
python3 run_chain.py smile3_psa -o psa_20261103_run01.root \
    --data-dir /mnt/d/UVSOR2026_late/killua/20261103_run01 \
    --set 'PSA_ReadData.extra_data_dirs=["/mnt/d/UVSOR2026_late/calltt/20261103_run01"]'
```

出力は `tutorials/cache/psa_cal_<ラン>.root` に置き、既にあれば流し直しません(PSA のチュートリアルと `4_MakeCalibration/PSA_Energy` で共有)。
`calib/psa` の較正ファイルを変えたときは、このファイルを消して流し直してください。

In [ ]:
def run_chain(out, *args):
    """run_chain.py を実行する(出力が既にあれば何もしない)"""
    cmd = [sys.executable, str(RUN_CHAIN), "smile3_psa", "-o", str(out), "--data-dir", str(RUN_DIRS[0]),
           "--set", "PSA_ReadData.extra_data_dirs=" + json.dumps([str(RUN_DIRS[1])]), *args]
    print(" ".join(cmd))
    if not out.exists():
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
        r.check_returncode()

OUT = CACHE / f"psa_cal_{RUN}.root"   # 今の calib/psa(位置・温度ゲイン・エネルギー)で流したもの
run_chain(OUT)

In [ ]:
def read_psa(path, columns):
    """PSA_TreeWriter/psa の列を numpy の配列で読む(short の列は int に直す)"""
    d = ROOT.RDataFrame("PSA_TreeWriter/psa", str(path)).AsNumpy(columns)
    out = {}
    for k, v in d.items():
        if v.dtype == object and len(v) and hasattr(v[0], "__len__"):
            out[k] = np.stack([np.asarray(x) for x in v])   # 配列の列 -> 2 次元
        elif v.dtype == object:
            out[k] = v.astype(int)
        else:
            out[k] = v
    return out

## 3. 位置の較正ファイル(64 点)

ha03 の `position_ha03.csv` の 64 点を、このランのバタフライマップ(4 節の条件で選んだヒット、粗いビン)に重ねます。数字は (n, m) です。
64 点は 8 列 × 8 行の格子として取れています(1 ラン分なので、バタフライマップの点の集まりはまだ粗い)。

In [ ]:
import csv
CAL = REPO / "calib" / "psa"
lines = [l for l in open(CAL / "position_ha03.csv") if not l.startswith("#")]
pos = list(csv.DictReader(lines))
print(lines[:3])
f = ROOT.TFile(str(OUT))
t = read_psa(OUT, ["board", "ch", "x_hg", "y_hg", "hg", "hg_corr", "x_corr", "y_corr", "pix_n", "pix_m", "pixel", "energy", "peak_id", "satu_h", "satu_l"])
good = (t["peak_id"] >= 26) & (t["peak_id"] <= 29) & (t["hg"] >= 30e3) & (t["satu_h"] == 0) & (t["satu_l"] == 0)
fig, axs = plt.subplots(2, 3, figsize=(17, 7))
for c, ax in enumerate(axs.flat):
    s = good & (t["board"] == 3) & (t["ch"] == c)
    ax.hist2d(t["x_hg"][s], t["y_hg"][s], bins=[120, 60], range=[[-0.32, 0.32], [-0.16, 0.16]], cmap="Greys")
    for r in pos:
        if int(r["ch"]) == c:
            ax.text(float(r["x"]), float(r["y"]), f"{r['n']}{r['m']}", color="r", fontsize=6, ha="center", va="center")
    ax.set_title(f"ha03 ch{c}: 64 peaks (n m)"); ax.set_xlabel("x_hg"); ax.set_ylabel("y_hg"); ax.set_aspect("equal")
fig.tight_layout(); plt.show()

## 4. 補正後の位置

`PSA_Calibration/Corrected_haNN_chC` は (`x_corr`, `y_corr`) の図です(`save_maps: true`)。
赤い線がピクセルの境目(半整数)で、点の集まりがマス目の中央に来ていれば補正がうまくいっています。
ここではツリーから、4 節と同じ条件のヒットで描きます。点の集まりがマス目に並んでいます(中央の列は縦に少しにじむ)。

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(16, 9.5))
for c, ax in enumerate(axs.flat):
    s = good & (t["board"] == 3) & (t["ch"] == c)
    ax.hist2d(t["x_corr"][s], t["y_corr"][s], bins=100, range=[[-0.5, 9.5], [-0.5, 9.5]])
    for b in np.arange(0.5, 9):
        ax.axvline(b, color="r", lw=0.4); ax.axhline(b, color="r", lw=0.4)
    ax.set_xlim(-0.5, 9.5); ax.set_ylim(-0.5, 9.5); ax.set_aspect("equal")
    ax.set_title(f"ha03 ch{c}"); ax.set_xlabel("x_corr (n)"); ax.set_ylabel("y_corr (m)")
fig.tight_layout(); plt.show()

## 5. ピクセルごとのヒットの数

ツリーの `pix_n`, `pix_m` から、ch ごとの 8 × 8 のヒットの数を描きます。

In [ ]:
s4 = good & (t["board"] == 3)
fig, axs = plt.subplots(1, 6, figsize=(20, 3.5))
for c, ax in enumerate(axs):
    m = s4 & (t["ch"] == c) & (t["pixel"] >= 0)
    occ = np.zeros((8, 8))
    np.add.at(occ, (t["pix_n"][m] - 1, t["pix_m"][m] - 1), 1)
    im = ax.imshow(occ.T, origin="lower", extent=[0.5, 8.5, 0.5, 8.5])
    ax.set_title(f"ha03 ch{c}"); ax.set_xlabel("n"); ax.set_ylabel("m"); fig.colorbar(im, ax=ax, fraction=0.045)
fig.tight_layout(); plt.show()

8 × 8 の中に入ったヒットの割合です(全部のヒットと、4 節の条件で選んだヒット)。
外に出たヒットの多くは、`hg` が小さいか時刻のずれたヒットです。ha01 の ch4(較正なし)と ha04(仮の較正)は低くなります。

In [ ]:
for b in np.unique(t["board"]):
    m = t["board"] == b
    per_ch = [100 * np.mean(t["pixel"][m & good & (t["ch"] == c)] >= 0) for c in range(6)]
    print(f"ha{b:02d}: in 8x8 {100 * np.mean(t['pixel'][m] >= 0):5.1f} % of all hits; selected hits per ch:",
          " ".join(f"{v:5.1f}" for v in per_ch))
m3 = t["board"] == 3
fig, ax = plt.subplots(figsize=(8, 3.8))
bins = np.logspace(1, 6, 100)
ax.hist(t["hg"][m3 & (t["pixel"] >= 0)], bins=bins, histtype="step", label="ha03 in 8x8")
ax.hist(t["hg"][m3 & (t["pixel"] < 0)], bins=bins, histtype="step", label="ha03 outside")
ax.set_xscale("log"); ax.set_xlabel("hg"); ax.set_ylabel("hits"); ax.legend()
plt.show()

## 6. ピクセルごとの hg スペクトル(エネルギーの較正の材料)

`PSA_Calibration/PixelHG_haNN` は、ピクセル(0〜383)× `hg_corr`(温度で直した hg)のヒストグラムです。
エネルギーの較正(`energy_haNN.csv` の `p0, p1`)は、線源のランでこのピクセルごとのスペクトルの光電ピークから作ります(`scripts/calib/make_psa_energy.py`)。
ha03 の ch0 から、いくつかのピクセルのスペクトルを描きます。

In [ ]:
a, ex, ey = th2(f.Get("PSA_Calibration/PixelHG_ha03"))
fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
m = ax[0].pcolormesh(ex, ey, a.T, norm=LogNorm(vmin=1), shading="flat")
ax[0].set_xlabel("pixel (64 ch + 8 (n-1) + m-1)"); ax[0].set_ylabel("hg_corr"); ax[0].set_title("PixelHG_ha03")
fig.colorbar(m, ax=ax[0])
for pix in [8 * 2 + 2, 8 * 3 + 4, 8 * 4 + 3, 8 * 5 + 5]:
    ax[1].step(ey[:-1], a[pix], where="post", label=f"ch0 n={pix // 8 + 1} m={pix % 8 + 1}")
ax[1].set_xlabel("hg_corr"); ax[1].set_ylabel("hits / bin"); ax[1].set_yscale("log"); ax[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

## 7. エネルギー

`energy_haNN.csv`(列 `ch,n,m,p0,p1`)が `energy_dir` にあるボードだけ、`energy` [keV] が入ります(無いボードと、ピクセルの無いヒットは −1)。
`PSA_Calibration/Energy_haNN` はボードごとのエネルギーのスペクトルです。Cs-137 のランなので、662 keV に光電ピークが出ます。

In [ ]:
print("energy files:", sorted(p.name for p in CAL.glob("energy_ha*.csv")))
fig, ax = plt.subplots(figsize=(10, 4))
for k in f.Get("PSA_Calibration").GetListOfKeys():
    if k.GetName().startswith("Energy_ha"):
        h = k.ReadObj()
        e = np.array([h.GetBinCenter(i) for i in range(1, h.GetNbinsX() + 1)])
        v = np.array([h.GetBinContent(i) for i in range(1, h.GetNbinsX() + 1)])
        ax.step(e, v, where="mid", lw=0.8, label=k.GetName()[7:])
ax.axvline(662, color="k", lw=0.5); ax.set_xlim(0, 1000)
ax.set_xlabel("energy [keV]"); ax.set_ylabel("hits / keV"); ax.legend(ncol=5, fontsize=7)
plt.show()
s = good & (t["board"] == 3) & (t["energy"] > 0)
print("ha03, selected hits: median energy", round(float(np.median(t["energy"][s])), 1), "keV")

## 8. パラメータ

In [ ]:
show_params("smile3_psa", "PSA_Calibration")